# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR⁲) Exploration with `mlcroissant`

This notebook demonstrates how to explore the FAIR⁲ dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/api_reference/) library in Python. Each entity (record set, field, column) is referenced by its Croissant `@id`.

### Dataset Source
The dataset is defined by the following Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```
It contains detailed clinicopathological and molecular records of second primary colorectal cancer in cancer survivors.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Croissant schema URL for the FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata  # Metadata is a Metadata object (not a dict)

print(f"{meta.name}:\n{meta.description}")

## 2. Data Overview
Explore available **record sets** and their **fields** (`@id` references).

We will print:
- Each record set `@id` and its name.
- For each record set, its fields with their `@id` and name.

In [ ]:
# Inspect available record sets and fields
record_sets = meta.record_sets

if not record_sets:
    print("No record sets are defined in the metadata.")
else:
    for rs in record_sets:
        print(f"\nRecordSet @id: {rs.id}\n  Name: {rs.name}")
        if hasattr(rs, 'fields') and rs.fields:
            for field in rs.fields:
                print(f"    Field @id: {field.id:50s} Name: {field.name}")
        else:
            print("    No fields found.")

## 3. Data Extraction
Load data from a specific record set as a DataFrame, using only the `@id` references for the record set and fields.

Below, we will extract records from the main record set, which in this dataset is commonly called `'patient'` or similar (inspect the above output for precise `@id`).

In [ ]:
# Identify all available record set @ids
record_set_ids = [rs.id for rs in meta.record_sets]
print('Available record set @ids:')
for rsi in record_set_ids:
    print('  ', rsi)

# For demonstration, use the first record set (most FAIR^2 datasets have one primary set)
main_record_set_id = record_set_ids[0]
print(f"\nLoading records from: {main_record_set_id}")

records = list(dataset.records(record_set=main_record_set_id))

df = pd.DataFrame(records)
print(f"Available columns (field @id): {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform some common EDA operations on the tabular data. We'll:
- Select a numeric field by its `@id`.
- Filter to include only records with high values.
- Normalize the selected field.
- Group by a key categorical field (again using `@id`).

In [ ]:
# List available fields
print('Available fields:')
for col in df.columns:
    print(f'  {col}')

# For demo, let's pick likely fields by @id (replace as appropriate after inspecting fields):
# Let's suppose '@id' 'age' is a numeric field & '@id' 'sex' is a categorical field.
# Replace below with exact field @ids observed above. For illustration, we'll attempt common ones.
# If not present, change as needed after running previous cells.
possible_numeric_ids = [col for col in df.columns if 'age' in col or 'interval' in col]
if possible_numeric_ids:
    numeric_field_id = possible_numeric_ids[0]
else:
    numeric_field_id = df.columns[0]  # fallback (likely not numeric)

print(f"Chosen numeric field @id: {numeric_field_id}")

try:
    numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
    df[numeric_field_id] = numeric_series
except Exception as e:
    print(f"Warning: Could not convert {numeric_field_id} to numeric.")

threshold = numeric_series.quantile(0.8) if numeric_series.notna().sum() > 0 else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print("\nNormalized field:")
print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Try grouping by a likely categorical field
possible_group_ids = [col for col in df.columns if ('sex' in col or 'MSI' in col or 'comorbidity' in col)]
if possible_group_ids:
    group_field_id = possible_group_ids[0]
    print(f"\nGrouping by field @id: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(grouped_df.head())
else:
    print("No suitable group field found to run groupby().")

## 5. Visualization
Basic distributions and relationships between selected fields. We recommend using field `@id` for all references.

For demonstration, we'll visualize the distribution of the selected numeric field, and if available, compare across a categorical variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f'Distribution of field {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.show()

# If group_field_id exists, boxplot by group
if 'group_field_id' in locals():
    plt.figure(figsize=(8,5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.show()

## 6. Conclusion

- We explored the FAIR⁲ dataset loaded via its Croissant schema using the `mlcroissant` API.
- Entities such as record sets and fields were referenced via their `@id` for reliable downstream analysis.
- After data extraction, initial EDA, normalization, grouping, and visualization were performed on selected fields.
- The notebook can be extended for in-depth clinical or biomarker analysis using detailed domain knowledge.

For further Croissant metadata references and programmatic dataset access patterns, consult the [`mlcroissant` documentation](https://mlcommons.github.io/croissant/) and FAIR⁲ dataset guides.